# OpenAlex corpus

## Setup

In [ ]:
from pathlib import Path
import os

import requests
import yaml
from dotenv import load_dotenv

def get_config(path):
    path = Path(path)
    with path.open(encoding="utf-8") as file:
        config = yaml.safe_load(file)
    return config

config = get_config("../config/config.yaml")
corpus = config["corpus"]

load_dotenv()
config["api_key"] = os.environ["OPENALEX_API_KEY"]

## Query

In [ ]:
works_url = "https://api.openalex.org/works"

subfields = "|".join(str(subfield) for subfield in corpus["subfields"])

filters = ",".join([
    f"publication_year:>{corpus['min_year'] - 1}",  # filter is strict >, so -1 keeps min_year in
    f"has_abstract:{str(corpus['has_abstract']).lower()}",
    f"primary_topic.subfield.id:{subfields}",
])

params = {
    "filter": filters,
    "select": ",".join(corpus["cols_to_select"]),
    "api_key": config["api_key"],
}

## Helpers

In [ ]:
def paginate(url, params=None, per_page=200):
    """Yield successive pages from a cursor-paginated OpenAlex endpoint."""
    params = dict(params or {})
    params["per-page"] = per_page
    cursor = "*"

    while cursor:
        params["cursor"] = cursor
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        page = response.json()

        if not page.get("results"):
            break

        yield page
        cursor = page.get("meta", {}).get("next_cursor")

In [ ]:
def reconstruct_abstract(inverted_abstract):
    abstract = []
    for word, positions in inverted_abstract.items():
        for position in positions:
            while len(abstract) <= position:
                abstract.append(None)
            abstract[position] = word
    return " ".join(abstract)

## Sanity checks on one page

In [ ]:
pages = paginate(works_url, params, per_page=25)
first_page = next(pages)
print("Number of results:", first_page["meta"]["count"])

In [ ]:
for work in first_page["results"]:
    print(work["title"])

In [ ]:
print(first_page["results"][0].keys())

In [ ]:
[reconstruct_abstract(work["abstract_inverted_index"]) for work in first_page["results"][:5]]
# All abstracts read normally, seems to work okay